# UniSweet raw/P&L-aligned report visuals

**Audience:** UniSweet Leadership and Finance Business Partners  
**Use:** English, annotation-rich components for distributed executive slides  
**Big Idea:** Topline and share deteriorated - concentrated in OLIVE customer x SKU losses and H2 - while lower marketing expenditure temporarily protected PBO; management should selectively reinvest to restore profitable growth.

Internal Sales and P&L compare **FY2024 with FY2023**. Market data is a separate **MAT Nov'24 versus MAT-1** view. All Sales rows are retained so the Sales headlines reconcile exactly to P&L; rows flagged `TURNOVER_GT_GSV` remain subject to business validation.

In [1]:
from pathlib import Path
import importlib.util
import os
import sys
import textwrap
from xml.etree import ElementTree

os.environ.setdefault("MPLCONFIGDIR", "/tmp/unisweet-mpl-cache")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/unisweet-xdg-cache")

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import font_manager
from matplotlib.patches import FancyBboxPatch
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts" / "storyline_metrics.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate scripts/storyline_metrics.py")


PROJECT_ROOT = find_project_root()
METRICS_PATH = PROJECT_ROOT / "scripts" / "storyline_metrics.py"
spec = importlib.util.spec_from_file_location("storyline_metrics", METRICS_PATH)
storyline_metrics = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(storyline_metrics)
D = storyline_metrics.STORYLINE_DATAFRAMES

# The storytelling-with-data skill is the source of truth for the visual
# system. It is imported here, not in the appendix, so save_component() can
# lint every report figure as it is exported. Both modules force the Agg
# backend on import, so hand the notebook's own backend back afterwards.
SKILL_DIR = PROJECT_ROOT / ".claude" / "skills" / "storytelling-with-data"
for path in (str(SKILL_DIR), str(PROJECT_ROOT / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

_NOTEBOOK_BACKEND = matplotlib.get_backend()
import lint
import swd
matplotlib.use(_NOTEBOOK_BACKEND, force=True)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "report_visuals"
DATA_DIR = OUTPUT_DIR / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Muted two-ramp visual system, shared with swd.SEQ_BLUE / swd.SEQ_ORANGE.
# BLUE MARKS FAVOURABLE MOVEMENT, ORANGE MARKS ADVERSE MOVEMENT. Favourable
# means good for UniSweet, which is not always a positive number: a €15.5m cut
# in marketing spend is blue. Direction is never left to colour alone - the
# printed sign says it, and a hatch marks every bar whose value is positive.
PRIMARY = "#004c6d"       # totals and anchors, headline, panel titles
FOCUS = "#346888"
SECONDARY = "#5886a5"     # favourable movement
SUPPORT = "#7aa6c2"
CONTEXT = "#9dc6e0"       # fills for favourable data that is context
PALE = "#c1e7ff"          # favourable washes and card fills

CONTRA = "#f79545"        # adverse movement - the losses that are the story
CONTRA_MID = "#faa35e"
CONTRA_SOFT = "#fdb177"   # adverse, but context rather than signal
CONTRA_PALE = "#ffdbc1"   # adverse washes and card fills

# No step of the orange ramp reaches 4.5:1 against white or against its own
# pale tint (#f79545 manages 2.3:1), so orange is a fill and mark colour only.
# Text that sits on or beside an orange fill is INK.

# Grey is what everything that is NOT the story is made of. Only data the
# headline actually argues about gets a hue; anchors, sums, offsets and
# not-this-half context recede into the page. Two greys because fills and text
# need different weights: a #8c8c8c fill reads as background, but #8c8c8c text
# is only 3.1:1 on white, so muted TEXT is MID.
INK = "#262626"           # text that must be read
MID = "#595959"           # secondary and de-emphasised TEXT
BASE = "#8c8c8c"          # context data: present, not competing
MUTED = "#bfbfbf"         # totals and anchors; zero lines; the source line
RULE = "#d9d9d9"          # the one remaining spine
WHITE = "#ffffff"

ARIAL_PATH = font_manager.findfont("Arial", fallback_to_default=False)
plt.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "figure.facecolor": WHITE,
    "axes.facecolor": WHITE,
    "text.color": INK,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "xtick.color": MID,
    "ytick.color": MID,
    "axes.edgecolor": RULE,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "svg.fonttype": "none",
})

MANIFEST = []


def new_figure(*, nrows=1, ncols=1, gridspec_kw=None):
    return plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(40 / 3, 7.5),
        gridspec_kw=gridspec_kw,
    )


def add_header(fig, title, subtitle):
    # The headline owns the top-left corner, where the eye enters. Long claims
    # shrink rather than run into the right margin - the Z sweep needs the
    # margin to stay a margin.
    fontsize = min(23.0, 23.0 * 70 / max(len(title), 1))
    fig.suptitle(title, x=0.055, y=0.965, ha="left", fontsize=fontsize, fontweight="bold", color=PRIMARY)
    fig.text(0.055, 0.915, subtitle, ha="left", va="top", fontsize=11.5, color=MID)


def add_footer(fig, source, caveat=""):
    footer = f"Source: {source}"
    if caveat:
        footer += f"  |  {caveat}"
    # Present for provenance, never competing: the source line is the faintest
    # thing on the page.
    fig.text(0.055, 0.025, footer, ha="left", va="bottom", fontsize=8.5, color=MUTED)


def add_callout(ax, x, y, text, color=PRIMARY, fontsize=13, **kwargs):
    """Narrative text, always read from a left edge (the Z path).

    x/y are axes fractions, so a callout keeps its place when limits change.
    Labels attached to a mark are the exception and stay on their mark.
    """
    return ax.text(x, y, text, transform=ax.transAxes, ha="left", va="top",
                   fontsize=fontsize, color=color, fontweight="bold", **kwargs)


def clean_axis(ax, baseline=False):
    """No gridlines, no border, no tick marks.

    Once every mark carries a direct label the value axis goes with the
    gridlines - a tick that repeats a printed number is the same clutter.
    `baseline=True` keeps the bottom spine for the waterfalls, whose bars are
    anchored on it; every other chart draws its own explicit zero line.
    """
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_visible(baseline)
    if baseline:
        ax.spines["bottom"].set_color(RULE)
    ax.grid(False)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)


def signed(value, decimals=1, suffix=""):
    return f"{value:+,.{decimals}f}{suffix}"


def save_component(visual_id, page, title, fig, data, source, caveat=""):
    data_path = DATA_DIR / f"{visual_id}.csv"
    png_path = OUTPUT_DIR / f"{visual_id}.png"
    svg_path = OUTPUT_DIR / f"{visual_id}.svg"
    data.to_csv(data_path, index=False)
    fig.savefig(png_path, dpi=144, facecolor=WHITE)
    fig.savefig(svg_path, facecolor=WHITE)
    # Advisory, never fatal: chart 03 is a contribution bridge drawn from a
    # 58.0 pp baseline, so ZERO_BASELINE fires there by design. That truncation
    # is stated in the chart's own caveat line rather than silently redrawn.
    findings = lint.check(fig)
    for finding in findings:
        print(f"lint {visual_id}: {finding}")
    MANIFEST.append({
        "visual_id": visual_id,
        "storyline_page": page,
        "action_title": title,
        "png": png_path.relative_to(PROJECT_ROOT).as_posix(),
        "svg": svg_path.relative_to(PROJECT_ROOT).as_posix(),
        "data_csv": data_path.relative_to(PROJECT_ROOT).as_posix(),
        "source": source,
        "caveat": caveat,
        "lint": "; ".join(str(f) for f in findings) or "clean",
    })
    plt.show()


def draw_waterfall(ax, frame, value_col, label_formatter, baseline=0.0):
    """Adverse steps orange, favourable steps blue - and the totals in grey.

    The opening and closing bars are the tallest shapes on the page but they
    are only the frame: the story is the steps between them. Grey stops that
    area from spending the reader's attention. Positive steps also carry a
    hatch, so the split survives a reader who cannot separate the two hues.
    """
    values = frame[value_col].astype(float).to_numpy()
    kinds = frame["kind"].tolist()
    steps = frame["step"].tolist()
    running = values[0]
    endpoints = []
    for i, (step, kind, value) in enumerate(zip(steps, kinds, values)):
        if kind == "total":
            bottom = baseline
            height = value - baseline
            color = MUTED
            hatch = None
            endpoint = value
        else:
            next_value = running + value
            bottom = min(running, next_value)
            height = abs(value)
            color = SECONDARY if value >= 0 else CONTRA
            hatch = "///" if value >= 0 else None
            endpoint = next_value
            running = next_value
        ax.bar(i, height, bottom=bottom, width=0.62, color=color,
               edgecolor=RULE if kind == "total" else PRIMARY, linewidth=0.8, hatch=hatch)
        endpoints.append(endpoint)
        label_y = max(bottom + height, endpoint)
        if kind == "change" and value < 0:
            label_y = bottom - 0.025 * max(abs(values).max(), 1)
            va = "top"
        else:
            label_y = label_y + 0.025 * max(abs(values).max(), 1)
            va = "bottom"
        ax.text(i, label_y, label_formatter(value, kind), ha="center", va=va, fontsize=11,
                fontweight="bold" if kind == "change" else "normal",
                color=INK if kind == "change" else MID)
        if i < len(values) - 1:
            connector_level = endpoint
            ax.plot([i + 0.31, i + 0.69], [connector_level, connector_level], color=MUTED, linewidth=1)
    ax.set_xticks(range(len(steps)), [step.replace(" ", "\n", 1) for step in steps], fontsize=10)
    ax.set_yticks([])
    return endpoints


print(f"Arial font: {ARIAL_PATH}")
print(f"Raw storyline metrics: {METRICS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Visual system: swd.ACCENT {swd.ACCENT} / swd.ACCENT_2 {swd.ACCENT_2}")


Arial font: /System/Library/Fonts/Supplemental/Arial.ttf
Raw storyline metrics: scripts/storyline_metrics.py
Visual system: swd.ACCENT #004c6d / swd.ACCENT_2 #f79545


## Validation before charting

In [2]:
reconciliation = D["pnl_reconciliation"]
check_columns = [column for column in reconciliation if column.startswith("check_")]
assert np.allclose(reconciliation[check_columns], 0.0, atol=1e-8)

topline_bridge = D["topline_bridge"]
assert np.isclose(
    topline_bridge.loc[0, "value_keur"] + topline_bridge.loc[1:2, "value_keur"].sum(),
    topline_bridge.loc[3, "value_keur"],
)
share_bridge = D["market_share_bridge"]
assert np.isclose(
    share_bridge.loc[0, "value_pp"] + share_bridge.loc[1:4, "value_pp"].sum(),
    share_bridge.loc[5, "value_pp"],
)
pbo_bridge = D["pbo_bridge"]
assert np.isclose(
    pbo_bridge.loc[0, "value_keur"] + pbo_bridge.loc[1:2, "value_keur"].sum(),
    pbo_bridge.loc[3, "value_keur"],
)

display(reconciliation.round(6))
display(D["quality"])

,reporting_year,gsv_keur,turnover_keur,discount_keur,pnl_gsv_keur,check_gsv_keur,pnl_turnover_keur,check_turnover_keur,pnl_discount_keur,check_discount_keur
0,2023,371528.3,290066.9,81461.4,371528.3,-0.0,290066.9,0.0,81461.4,-0.0
1,2024,349247.6,270504.3,78743.3,349247.6,0.0,270504.3,-0.0,78743.3,0.0


,metric,value
0,Raw Sales rows,6953.0
1,Flagged TO > GSV TO FY2023,2574.5
2,Flagged TO > GSV TO FY2024,3173.0


## Page 1 - Executive truth

In [3]:
# Four cards, not eight: the exact chain the headline claims. GSV, Discount/GSV,
# Gross Margin and PBO Margin are the subject of charts 02 and 09, not page 1.
CARD_METRICS = ["Turnover", "Gross Profit", "Marketing Expense", "PBO"]
# Adverse for UniSweet, not simply negative: the -€15.5m marketing line is the
# lever that protected PBO, so it reads blue alongside PBO itself.
ADVERSE = {"Turnover", "Gross Profit"}
cards = D["headline"].set_index("metric").loc[CARD_METRICS].reset_index()

fig, axes = new_figure(nrows=1, ncols=4)
fig.subplots_adjust(left=0.05, right=0.97, bottom=0.32, top=0.83, wspace=0.08)
title = "Topline weakened while lower marketing spend protected PBO"
add_header(fig, title, "FY2024 versus FY2023, EURm | Raw Sales/P&L-aligned reporting base")

for ax, row in zip(axes.flat, cards.itertuples(index=False)):
    ax.set_axis_off()
    adverse = row.metric in ADVERSE
    accent = CONTRA if adverse else PRIMARY
    face = CONTRA_PALE if adverse else PALE
    ax.add_patch(FancyBboxPatch(
        (0.01, 0.02), 0.98, 0.94, boxstyle="round,pad=0.012,rounding_size=0.03",
        transform=ax.transAxes, facecolor=face, edgecolor=accent, linewidth=1.5))
    ax.text(0.08, 0.82, row.metric.upper(), transform=ax.transAxes,
            fontsize=11, color=MID, fontweight="bold")
    # The fill and border carry the verdict; the number stays INK because no
    # step of the orange ramp is readable as text on its own pale tint.
    ax.text(0.08, 0.46, f"€{getattr(row, 'value_2024') / 1_000:,.1f}m",
            transform=ax.transAxes, fontsize=34, color=INK, fontweight="bold")
    ax.text(0.08, 0.16, f"{row.absolute_change / 1_000:+,.1f}m YoY",
            transform=ax.transAxes, fontsize=14, color=INK, fontweight="bold")

fig.text(0.055, 0.22,
         "The €15.5m marketing cut is larger than the €10.3m of Gross Profit lost - "
         "PBO is up €5.2m on cost, not on growth.",
         ha="left", va="top", fontsize=14, color=PRIMARY, fontweight="bold")

source = "outputs/sales_master.csv and P&L Table.xlsx"
caveat = "All raw Sales rows included; flagged TO > GSV rows remain subject to validation"
add_footer(fig, source, caveat)
save_component("01_executive_scorecard", 1, title, fig, cards, source, caveat)


<Figure size 1333.33x750 with 4 Axes>

In [4]:
bridge = D["topline_bridge"].copy()
fig, ax = new_figure()
fig.subplots_adjust(left=0.07, right=0.96, bottom=0.15, top=0.80)
title = "Lower GSV explains €17.40m of the TO decline; discount intensity explains €2.17m"
add_header(fig, title,
           "Turnover, EURm | FY2024 versus FY2023 | The two effects reconcile the €19.56m TO gap")
draw_waterfall(
    ax,
    bridge,
    "value_keur",
    lambda value, kind: f"€{value / 1_000:,.1f}m" if kind == "total" else f"{value / 1_000:+.2f}m",
)
ax.set_ylim(0, 360_000)
clean_axis(ax, baseline=True)
add_callout(ax, 0.01, 0.99, "~89% of the decline\ntracks to the lower GSV base")
source = "outputs/sales_master.csv"
caveat = "Rate-hold arithmetic; not a measured promotion ROI or causal estimate"
add_footer(fig, source, caveat)
save_component("02_turnover_decline_bridge", 1, title, fig, bridge, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

## Page 2 - Market diagnosis

In [5]:
# Drawn from zero. The old form was a share bridge floating on a 58.0 pp
# baseline, which truncated every bar length. On a true zero axis the share
# LEVELS (64.65 -> 62.68) leave the contributions as 4 px hairlines, so the
# chart plots the contributions themselves - each bar read from zero, at full
# size - and carries the two levels in the subtitle where they belong.
bridge = D["market_share_bridge"].copy()
levels = bridge.loc[bridge["kind"] == "total", "value_pp"].tolist()
moves = bridge.loc[bridge["kind"] == "change"].reset_index(drop=True)
net = moves["value_pp"].sum()

fig, ax = new_figure()
fig.subplots_adjust(left=0.07, right=0.96, bottom=0.17, top=0.80)
title = "OLIVE erased SKY and COBALT gains, taking UniSweet share down 1.97 points"
add_header(fig, title,
           f"Brand contribution to UniSweet value share, points | MAT Nov'24 versus MAT-1 | "
           f"Total share {levels[0]:.2f}% → {levels[1]:.2f}%")

labels = moves["step"].tolist() + ["Net change"]
values = moves["value_pp"].tolist() + [net]
x = list(range(len(moves))) + [len(moves) + 0.6]
# Grey for the two bars that are not brand movements: the source-rounding
# artefact kept only for transparency, and the net, which is just the sum of
# the bars beside it.
context = [step == "Source rounding" for step in moves["step"]] + [True]
noise = [step == "Source rounding" for step in moves["step"]] + [False]
colors = [BASE if ctx else (SECONDARY if value >= 0 else CONTRA)
          for value, ctx in zip(values, context)]
bars = ax.bar(x, values, width=0.62,
              color=colors,
              edgecolor=[RULE if ctx else PRIMARY for ctx in context],
              linewidth=[0.8] * len(moves) + [2.0])
for bar, value, ctx in zip(bars, values, context):
    if value >= 0 and not ctx:
        bar.set_hatch("///")
ax.axhline(0, color=MUTED, linewidth=1)
ax.set_xticks(x, [label.replace(" ", "\n", 1) for label in labels], fontsize=10)
for tick, quiet in zip(ax.get_xticklabels(), noise):
    tick.set_color(MUTED if quiet else MID)
ax.set_yticks([])
ax.set_ylim(-3.7, 1.5)
# The net bar is grey because it is a sum, not a fifth brand - but its label
# is the number in the headline, so it stays bold.
for xi, value, quiet in zip(x, values, noise):
    ax.text(xi, value + (0.07 if value >= 0 else -0.07), f"{value:+.2f} pp",
            ha="center", va="bottom" if value >= 0 else "top",
            fontsize=11, fontweight="normal" if quiet else "bold",
            color=MUTED if quiet else INK)
clean_axis(ax)
add_callout(ax, 0.01, 0.99, "SKY + COBALT offset +1.01 points;\nOLIVE alone gave back 2.98")
source = "Market Report MAT Nov'24.xlsx"
caveat = "Contributions read from zero; source rounding of -0.001 pp shown explicitly"
add_footer(fig, source, caveat)
save_component("03_market_share_bridge", 2, title, fig, bridge, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

In [6]:
channel = D["channel"].copy().set_index("channel").loc[["DT", "MT"]].reset_index()
fig, axes = new_figure(nrows=1, ncols=2, gridspec_kw={"width_ratios": [1, 1]})
fig.subplots_adjust(left=0.09, right=0.96, bottom=0.15, top=0.80, wspace=0.44)
title = "MT is the relative hotspot; DT still carries the larger internal value gap"
add_header(fig, title, "Internal Sales: FY2024 vs FY2023 | Market: MAT Nov'24 vs MAT-1")

y = np.arange(len(channel))
colors = [CONTRA_SOFT, CONTRA]   # both channels adverse; MT is the hotspot

bars = axes[0].barh(y, channel["to_change_keur"] / 1_000, color=colors, height=0.42)
axes[0].axvline(0, color=MUTED, linewidth=1)
axes[0].set_ylim(-0.75, 1.75)
axes[0].set_yticks(y, channel["channel"], fontsize=12, fontweight="bold")
axes[0].set_xticks([])
axes[0].set_title("DT has the larger absolute TO gap, EURm", loc="left", fontsize=14,
                  color=PRIMARY, fontweight="bold")
for bar, value in zip(bars, channel["to_change_keur"] / 1_000):
    axes[0].text(value / 2, bar.get_y() + bar.get_height() / 2, f"{value:+.2f}m",
                 va="center", ha="center", color=INK, fontsize=13, fontweight="bold")
axes[0].set_xlim(-13.5, 1.0)
clean_axis(axes[0])

bars = axes[1].barh(y, channel["share_movement_pp"], color=colors, height=0.42)
axes[1].axvline(0, color=MUTED, linewidth=1)
axes[1].set_ylim(-0.75, 1.75)
axes[1].set_yticks(y, channel["channel"], fontsize=12, fontweight="bold")
axes[1].set_xticks([])
axes[1].set_title("MT has the much larger relative share loss, points", loc="left", fontsize=14,
                  color=PRIMARY, fontweight="bold")
for bar, row in zip(bars, channel.itertuples(index=False)):
    if abs(row.share_movement_pp) > 1:
        label_x, label_ha = row.share_movement_pp / 2, "center"
    else:
        label_x, label_ha = row.share_movement_pp - 0.10, "right"
    axes[1].text(label_x, bar.get_y() + bar.get_height() / 2, f"{row.share_movement_pp:+.2f} pp",
                 va="center", ha=label_ha, color=INK, fontsize=13, fontweight="bold")
    # Who took it: supporting evidence, not the claim. Kept, but pushed back
    # far enough that it reads only after the share movement it explains.
    axes[1].text(0.10, bar.get_y() + bar.get_height() / 2,
                 f"{row.competitor} {row.competitor_share_movement_pp:+.2f} pp",
                 va="center", ha="left", color=MID, fontsize=9)
axes[1].set_xlim(-6.5, 1.6)
clean_axis(axes[1])

source = "Raw Sales and Market Report MAT Nov'24.xlsx"
caveat = "FY and MAT periods are shown side by side, not reconciled"
add_footer(fig, source, caveat)
save_component("04_channel_hotspot", 2, title, fig, channel, source, caveat)


<Figure size 1333.33x750 with 2 Axes>

## Page 3 - Sales drivers

In [7]:
monthly = D["monthly"].copy()
fig, ax = new_figure()
fig.subplots_adjust(left=0.06, right=0.97, bottom=0.15, top=0.80)
title = "96% of the FY turnover decline occurred in H2"
add_header(fig, title, "Monthly TO change, EURm | FY2024 versus the same month in FY2023")
x = np.arange(12)
values = monthly["to_change_keur"] / 1_000
# The headline is about H2, so H1 is context and goes grey. Colour is spent
# only on the six months that carry 96% of the decline.
is_h2 = monthly["half"].eq("H2").tolist()
bar_colors = [(SECONDARY if value >= 0 else CONTRA) if h2 else BASE
              for value, h2 in zip(values, is_h2)]
bars = ax.bar(x, values, color=bar_colors,
              edgecolor=[PRIMARY if h2 else RULE for h2 in is_h2], linewidth=0.6)
for bar, value, h2 in zip(bars, values, is_h2):
    if value >= 0:
        bar.set_hatch("///")
ax.axhline(0, color=MUTED, linewidth=1)
ax.axvspan(5.5, 11.5, color=CONTRA_PALE, alpha=0.5, zorder=-1)   # H2 is the adverse half
for i, (value, h2) in enumerate(zip(values, is_h2)):
    va = "bottom" if value >= 0 else "top"
    offset = 0.18 if value >= 0 else -0.18
    ax.text(i, value + offset, f"{value:+.1f}", ha="center", va=va, fontsize=9.5,
            fontweight="bold" if h2 else "normal", color=INK if h2 else MID)
ax.set_xticks(x, ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
for tick, h2 in zip(ax.get_xticklabels(), is_h2):
    tick.set_color(INK if h2 else MID)
ax.set_yticks([])
ax.set_ylim(-8.7, 8.6)
clean_axis(ax)
add_callout(ax, 0.01, 0.99, "H2 (shaded): -€18.72m\n95.7% of the FY decline")
source = "outputs/sales_master.csv"
caveat = "Monthly sell-in pattern does not establish holiday or seasonal causality"
add_footer(fig, source, caveat)
save_component("05_monthly_to_change", 3, title, fig, monthly, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

In [8]:
customer = D["customer"].copy()
priority_names = ["Bliss", "Candies", "Macarons"]
priority = customer.set_index("customer_name").loc[priority_names].reset_index()
total_change = D["monthly"]["to_change_keur"].sum()
other_change = total_change - priority["to_change_keur"].sum()
chart = pd.concat([
    priority[["customer_name", "channel_code", "to_change_keur", "to_growth_pct"]],
    pd.DataFrame({"customer_name": ["All other customers (net)"], "channel_code": ["Mixed"], "to_change_keur": [other_change], "to_growth_pct": [np.nan]}),
], ignore_index=True)
chart = chart.sort_values("to_change_keur", ascending=False)

fig, ax = new_figure()
fig.subplots_adjust(left=0.22, right=0.95, bottom=0.15, top=0.80)
title = "Bliss, Candies and Macarons explain 91% of the turnover decline"
add_header(fig, title, "TO change, EURm | FY2024 versus FY2023 | Existing-account spend, not broad customer attrition")
y = np.arange(len(chart))
# "All other" is the 9% the headline sets aside - grey, so the three named
# accounts own the page.
is_other = [name.startswith("All other") for name in chart["customer_name"]]
colors = [BASE if other else CONTRA for other in is_other]
bars = ax.barh(y, chart["to_change_keur"] / 1_000, color=colors, height=0.58)
ax.axvline(0, color=MUTED, linewidth=1)
ax.set_yticks(y, [f"{name}  |  {channel}" for name, channel in zip(chart["customer_name"], chart["channel_code"])], fontsize=11)
ax.set_xticks([])
ax.set_ylim(-0.6, 4.3)
for bar, value, other in zip(bars, chart["to_change_keur"] / 1_000, is_other):
    # Long bars carry the label inside, the short context bar outside. INK
    # either way - orange is not dark enough to take white text.
    inside = value < -2.2
    ax.text(value + 0.15 if inside else value - 0.15, bar.get_y() + bar.get_height() / 2,
            f"{value:+.2f}m", va="center", ha="left" if inside else "right",
            color=MID if other else INK, fontsize=11,
            fontweight="normal" if other else "bold")
for tick, other in zip(ax.get_yticklabels(), is_other):
    tick.set_color(MUTED if other else INK)
clean_axis(ax)
active_2023 = int((customer["turnover_keur_2023"] > 0).sum())
active_2024 = int((customer["turnover_keur_2024"] > 0).sum())
add_callout(ax, 0.01, 0.99,
            f"Active customers: {active_2023} → {active_2024}\nOnly Treats exited (~€0.5k TO)", fontsize=12)
source = "outputs/sales_master.csv"
caveat = "Customer counts reflect observed UniSweet accounts, not market distribution"
add_footer(fig, source, caveat)
save_component("06_customer_concentration", 3, title, fig, chart, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

In [9]:
product = D["product"].copy()
declines = product.nsmallest(4, "to_change_keur")
gains = product.nlargest(3, "to_change_keur")
selected_names = set(declines["product_name"]) | set(gains["product_name"])
other = product.loc[~product["product_name"].isin(selected_names), "to_change_keur"].sum()
chart = pd.concat([
    declines[["product_name", "to_change_keur", "to_growth_pct"]],
    gains[["product_name", "to_change_keur", "to_growth_pct"]],
    pd.DataFrame({"product_name": ["All other products (net)"], "to_change_keur": [other], "to_growth_pct": [np.nan]}),
], ignore_index=True).sort_values("to_change_keur")

fig, ax = new_figure()
fig.subplots_adjust(left=0.20, right=0.95, bottom=0.15, top=0.80)
title = "POUCH 900GR and PACK 1.1KG drive the loss, partly offset by growing formats"
add_header(fig, title, "TO change, EURm | FY2024 versus FY2023 | Exact-product change")
y = np.arange(len(chart))
# The named formats are the story; the residual "all other" line is not.
is_other = [name.startswith("All other") for name in chart["product_name"]]
colors = [BASE if other else (CONTRA if value < 0 else SECONDARY)
          for value, other in zip(chart["to_change_keur"], is_other)]
bars = ax.barh(y, chart["to_change_keur"] / 1_000, color=colors,
               edgecolor=[RULE if other else PRIMARY for other in is_other], linewidth=0.6)
for bar, value, other in zip(bars, chart["to_change_keur"], is_other):
    if value >= 0 and not other:
        bar.set_hatch("///")
ax.axvline(0, color=MUTED, linewidth=1)
ax.set_yticks(y, chart["product_name"], fontsize=11)
ax.set_xticks([])
for tick, other in zip(ax.get_yticklabels(), is_other):
    tick.set_color(MUTED if other else INK)
for bar, value, other in zip(bars, chart["to_change_keur"] / 1_000, is_other):
    # Losses take the label inside the orange bar, gains just past the blue one.
    ax.text(value + 0.18, bar.get_y() + bar.get_height() / 2, f"{value:+.2f}m",
            va="center", ha="left", color=MID if other else INK, fontsize=10.5,
            fontweight="normal" if other else "bold")
ax.invert_yaxis()   # the losses the headline names read first, top down
clean_axis(ax)
source = "outputs/sales_master.csv"
caveat = "Pack names identify formats; they do not support physical-volume inference"
add_footer(fig, source, caveat)
save_component("07_product_losses_offsets", 3, title, fig, chart, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

In [10]:
matrix_data = D["customer_sku"].copy()
customers = ["Bliss", "Candies", "Macarons"]
products = ["POUCH 900GR", "PACK 1.1KG", "POUCH 100GR", "POUCH 400GR"]
to_matrix = matrix_data.pivot(index="customer_name", columns="product_name", values="to_change_keur").reindex(index=customers, columns=products).fillna(0) / 1_000
discount_matrix = matrix_data.pivot(index="customer_name", columns="product_name", values="discount_pct_gsv_movement_bps").reindex(index=customers, columns=products)

fig, axes = new_figure(nrows=1, ncols=2)
fig.subplots_adjust(left=0.09, right=0.97, bottom=0.18, top=0.78, wspace=0.24)
title = "Customer x SKU losses require more than a broad discount response"
add_header(fig, title, "OLIVE priority accounts | FY2024 versus FY2023")

# Orange always marks the adverse movement. That is a FALLING turnover in the
# left panel and a RISING discount rate in the right one, so the two panels
# take opposite poles of the same diverging ramp.
adverse_low = swd.diverging_cmap(low=swd.SEQ_ORANGE[0], high=swd.SEQ_BLUE[0])
adverse_high = swd.diverging_cmap(low=swd.SEQ_BLUE[0], high=swd.SEQ_ORANGE[0])
for ax, matrix, cmap, panel_title, formatter in [
    (axes[0], to_matrix, adverse_low, "TO change (€m)", lambda value: f"{value:+.2f}"),
    (axes[1], discount_matrix, adverse_high, "Discount / GSV movement (bps)", lambda value: "n/a" if pd.isna(value) else f"{value:+.0f}"),
]:
    values = matrix.to_numpy(dtype=float)
    finite = np.abs(values[np.isfinite(values)])
    limit = finite.max() if len(finite) else 1
    norm = mcolors.TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)
    ax.imshow(values, cmap=cmap, norm=norm, aspect="auto")
    ax.set_title(panel_title, loc="left", fontsize=14, color=PRIMARY, fontweight="bold", pad=14)
    ax.set_xticks(range(len(products)), [label.replace(" ", "\n", 1) for label in products], fontsize=10)
    ax.set_yticks(range(len(customers)), customers, fontsize=11, fontweight="bold")
    for i in range(len(customers)):
        for j in range(len(products)):
            value = values[i, j]
            # White only on a deep blue cell; every orange step takes INK.
            deep = np.isfinite(value) and abs(value) > 0.45 * limit
            favourable = value > 0 if cmap is adverse_low else value < 0
            ax.text(j, i, formatter(value), ha="center", va="center", fontsize=11,
                    color=WHITE if (deep and favourable) else INK, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)

fig.text(0.055, 0.105,
         "Orange marks the adverse movement in each panel - falling turnover on the left, rising discount rate on the right. "
         "Printed signs and values carry direction on their own.",
         ha="left", fontsize=10, color=MID)
source = "outputs/sales_master.csv"
caveat = "Discount movement is diagnostic and does not establish promotion causality"
add_footer(fig, source, caveat)
save_component("08_customer_sku_discount_matrix", 3, title, fig, matrix_data, source, caveat)


<Figure size 1333.33x750 with 2 Axes>

## Page 4 - Profit bridge and actions

In [11]:
bridge = D["pbo_bridge"].copy()
pnl_total = D["pnl"].set_index("brand").loc["TOTAL"]
fig, ax = new_figure()
fig.subplots_adjust(left=0.07, right=0.96, bottom=0.16, top=0.80)
title = "€15.5m lower marketing spend more than offset €10.3m lower Gross Profit"
add_header(fig, title, "PBO, EURm | FY2024 versus FY2023 | PBO protection is cost-led, not growth-led")
draw_waterfall(
    ax,
    bridge,
    "value_keur",
    lambda value, kind: f"€{value / 1_000:,.1f}m" if kind == "total" else f"{value / 1_000:+.1f}m",
)
ax.set_ylim(0, 115_000)
clean_axis(ax, baseline=True)
add_callout(ax, 0.01, 0.99,
            f"PBO margin {pnl_total['pbo_margin_movement_bps']:+.0f} bps\n"
            f"on a €19.56m smaller topline")

source = "P&L Table.xlsx"
caveat = "Lower absolute supply-chain cost is not efficiency: Supply Chain Cost / TO worsened ~35 bps"
add_footer(fig, source, caveat)
chart_data = pd.concat([
    bridge.assign(section="PBO bridge"),
    pd.DataFrame({"step": ["PBO Margin"], "kind": ["rate"],
                  "value_keur": [pnl_total["pbo_margin_movement_bps"]],
                  "section": ["Margin movement, bps"]}),
], ignore_index=True)
save_component("09_pbo_bridge", 4, title, fig, chart_data, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

In [12]:
portfolio = D["portfolio"].copy().set_index("brand").loc[["OLIVE", "COBALT", "SKY"]].reset_index()
fig, ax = new_figure()
fig.subplots_adjust(left=0.12, right=0.95, bottom=0.16, top=0.80)
title = "COBALT is the scale proof point; SKY requires diagnosis before reinvestment"
add_header(fig, title,
           "Growth, % | Internal TO is FY2024 vs FY2023; market value is MAT Nov'24 vs MAT-1")
y = np.arange(len(portfolio))
internal = portfolio["internal_to_growth_pct"] * 100
market = portfolio["market_value_growth_pct"] * 100
# The headline names COBALT and SKY; OLIVE is the context they sit against.
# Shape says which measure (circle = internal, square = market); colour says
# whether that measure grew or shrank.
signal = [brand in {"COBALT", "SKY"} for brand in portfolio["brand"]]


def growth_fill(value, lit):
    if not lit:
        return BASE          # OLIVE is the backdrop the two named brands sit against
    return SECONDARY if value >= 0 else CONTRA


for yi, left, right, lit in zip(y, internal, market, signal):
    ax.plot([left, right], [yi, yi], color=SUPPORT if lit else RULE, linewidth=4, zorder=1)
edges = [PRIMARY if lit else RULE for lit in signal]
ax.scatter(internal, y, s=[150 if lit else 110 for lit in signal],
           color=[growth_fill(v, lit) for v, lit in zip(internal, signal)],
           edgecolor=edges, linewidth=0.8, marker="o", zorder=2)
ax.scatter(market, y, s=[165 if lit else 120 for lit in signal],
           color=[growth_fill(v, lit) for v, lit in zip(market, signal)],
           edgecolor=edges, linewidth=0.8, marker="s", zorder=3)
ax.axvline(0, color=MUTED, linewidth=1)
ax.set_yticks(y, portfolio["brand"], fontsize=12, fontweight="bold")
for tick, lit in zip(ax.get_yticklabels(), signal):
    tick.set_color(INK if lit else MUTED)
ax.set_xticks([])
ax.set_xlim(-15, 32)
ax.set_ylim(-0.55, 2.75)
for yi, i_value, m_value, lit in zip(y, internal, market, signal):
    # INK rather than the mark's own colour: orange is unreadable as text.
    ax.text(i_value - 0.7, yi - 0.13, f"Internal {i_value:+.1f}%", ha="right", va="bottom",
            fontsize=10.5, color=INK if lit else MID, fontweight="bold" if lit else "normal")
    ax.text(m_value + 0.7, yi + 0.13, f"Market {m_value:+.1f}%", ha="left", va="top",
            fontsize=10.5, color=INK if lit else MID, fontweight="bold" if lit else "normal")
# Each callout sits in the empty band under the row it speaks for, on the same
# left edge as the headline.
add_callout(ax, 0.01, 0.70, "SKY: reconcile sell-in, inventory and distribution first", fontsize=12)
add_callout(ax, 0.01, 0.40, "COBALT: scale where incremental PBO remains attractive", fontsize=12)
clean_axis(ax)
source = "outputs/sales_master.csv, P&L Table.xlsx and Market Report MAT Nov'24.xlsx"
caveat = "Different FY and MAT periods are juxtaposed for diagnosis, not direct reconciliation"
add_footer(fig, source, caveat)
save_component("10_portfolio_direction", 4, title, fig, portfolio, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

In [13]:
actions = D["action_cards"].copy()
fig, ax = new_figure()
fig.subplots_adjust(left=0.04, right=0.96, bottom=0.10, top=0.82)
ax.set_axis_off()
title = "Three gated actions restore profitable growth without giving back protected PBO"
add_header(fig, title, "Each action has one owner, one first gate and one financial outcome")

card_x = [0.02, 0.345, 0.67]
card_colors = [PRIMARY, FOCUS, SECONDARY]
for x0, color, row in zip(card_x, card_colors, actions.itertuples(index=False)):
    ax.add_patch(FancyBboxPatch((x0, 0.08), 0.29, 0.78, boxstyle="round,pad=0.015,rounding_size=0.025", transform=ax.transAxes, facecolor=WHITE, edgecolor=color, linewidth=2))
    ax.add_patch(FancyBboxPatch((x0, 0.70), 0.29, 0.16, boxstyle="round,pad=0.015,rounding_size=0.025", transform=ax.transAxes, facecolor=color, edgecolor=color, linewidth=0))
    ax.text(x0 + 0.025, 0.79, f"ACTION {row.action}", transform=ax.transAxes, color=WHITE, fontsize=11, fontweight="bold", va="center")
    ax.text(x0 + 0.025, 0.655, textwrap.fill(row.title, 29), transform=ax.transAxes, color=PRIMARY, fontsize=15, fontweight="bold", va="top", linespacing=1.15)
    ax.text(x0 + 0.025, 0.47, "OWNER", transform=ax.transAxes, color=MID, fontsize=9, fontweight="bold")
    ax.text(x0 + 0.025, 0.425, row.owner, transform=ax.transAxes, color=INK, fontsize=12, fontweight="bold")
    ax.text(x0 + 0.025, 0.34, "FIRST DECISION GATE", transform=ax.transAxes, color=MID, fontsize=9, fontweight="bold")
    ax.text(x0 + 0.025, 0.29, textwrap.fill(row.first_gate, 34), transform=ax.transAxes, color=INK, fontsize=11, va="top", linespacing=1.25)
    ax.text(x0 + 0.025, 0.17, "CORE KPI", transform=ax.transAxes, color=MID, fontsize=9, fontweight="bold")
    ax.text(x0 + 0.025, 0.12, textwrap.fill(row.core_kpi, 34), transform=ax.transAxes, color=INK, fontsize=11, fontweight="bold", va="top")

source = "calls-to-action-by-audience.md"
caveat = "Funding scales only after positive incremental Gross Profit and the agreed PBO/payback gate"
add_footer(fig, source, caveat)
save_component("11_action_cards", 4, title, fig, actions, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

## Export manifest and render QA

In [14]:
manifest = pd.DataFrame(MANIFEST)
manifest_path = OUTPUT_DIR / "visual_manifest.csv"
manifest.to_csv(manifest_path, index=False)

for row in manifest.itertuples(index=False):
    png = PROJECT_ROOT / row.png
    svg = PROJECT_ROOT / row.svg
    csv = PROJECT_ROOT / row.data_csv
    assert png.exists() and svg.exists() and csv.exists()
    with Image.open(png) as rendered:
        assert rendered.size == (1920, 1080), (row.visual_id, rendered.size)
    ElementTree.parse(svg)
    assert "Arial" in svg.read_text(encoding="utf-8")

quality_rubric = pd.DataFrame({
    "dimension": ["Audience", "Action", "Chart fit", "Accuracy", "Clutter", "Attention", "Accessibility", "Story", "Medium fit", "Polish"],
    "score": [2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
    "evidence": [
        "Leadership and FBP audience stated",
        "Action titles and decision implications",
        "Scorecard, bars, heatmaps and additive waterfalls",
        "Raw/P&L reconciliation and explicit period caveats",
        "No gridlines; direct labels replace every value axis",
        "Grey for context; hue only where the headline argues",
        "Signs, position, labels and hatches supplement colour",
        "Four-page logic from truth to action",
        "Annotations and sources support distributed slides",
        "Arial, one left edge for all narrative text, stable 16:9 exports",
    ],
})
quality_rubric.to_csv(OUTPUT_DIR / "quality_rubric.csv", index=False)
assert quality_rubric["score"].sum() >= 17

print(f"Exported {len(manifest)} visual components to {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")
print(f"Quality rubric: {quality_rubric['score'].sum()}/20")
display(manifest)
display(quality_rubric)


Exported 11 visual components to outputs/report_visuals
Quality rubric: 20/20


,visual_id,storyline_page,action_title,png,svg,data_csv,source,caveat,lint
0,01_executive_scorecard,1,Topline weakened while lower marketing spend p...,outputs/report_visuals/01_executive_scorecard.png,outputs/report_visuals/01_executive_scorecard.svg,outputs/report_visuals/data/01_executive_score...,outputs/sales_master.csv and P&L Table.xlsx,All raw Sales rows included; flagged TO > GSV ...,clean
1,02_turnover_decline_bridge,1,Lower GSV explains €17.40m of the TO decline; ...,outputs/report_visuals/02_turnover_decline_bri...,outputs/report_visuals/02_turnover_decline_bri...,outputs/report_visuals/data/02_turnover_declin...,outputs/sales_master.csv,Rate-hold arithmetic; not a measured promotion...,clean
2,03_market_share_bridge,2,"OLIVE erased SKY and COBALT gains, taking UniS...",outputs/report_visuals/03_market_share_bridge.png,outputs/report_visuals/03_market_share_bridge.svg,outputs/report_visuals/data/03_market_share_br...,Market Report MAT Nov'24.xlsx,Contributions read from zero; source rounding ...,clean
3,04_channel_hotspot,2,MT is the relative hotspot; DT still carries t...,outputs/report_visuals/04_channel_hotspot.png,outputs/report_visuals/04_channel_hotspot.svg,outputs/report_visuals/data/04_channel_hotspot...,Raw Sales and Market Report MAT Nov'24.xlsx,"FY and MAT periods are shown side by side, not...",clean
4,05_monthly_to_change,3,96% of the FY turnover decline occurred in H2,outputs/report_visuals/05_monthly_to_change.png,outputs/report_visuals/05_monthly_to_change.svg,outputs/report_visuals/data/05_monthly_to_chan...,outputs/sales_master.csv,Monthly sell-in pattern does not establish hol...,clean
5,06_customer_concentration,3,"Bliss, Candies and Macarons explain 91% of the...",outputs/report_visuals/06_customer_concentrati...,outputs/report_visuals/06_customer_concentrati...,outputs/report_visuals/data/06_customer_concen...,outputs/sales_master.csv,Customer counts reflect observed UniSweet acco...,clean
6,07_product_losses_offsets,3,"POUCH 900GR and PACK 1.1KG drive the loss, par...",outputs/report_visuals/07_product_losses_offse...,outputs/report_visuals/07_product_losses_offse...,outputs/report_visuals/data/07_product_losses_...,outputs/sales_master.csv,Pack names identify formats; they do not suppo...,clean
7,08_customer_sku_discount_matrix,3,Customer x SKU losses require more than a broa...,outputs/report_visuals/08_customer_sku_discoun...,outputs/report_visuals/08_customer_sku_discoun...,outputs/report_visuals/data/08_customer_sku_di...,outputs/sales_master.csv,Discount movement is diagnostic and does not e...,clean
8,09_pbo_bridge,4,€15.5m lower marketing spend more than offset ...,outputs/report_visuals/09_pbo_bridge.png,outputs/report_visuals/09_pbo_bridge.svg,outputs/report_visuals/data/09_pbo_bridge.csv,P&L Table.xlsx,Lower absolute supply-chain cost is not effici...,clean
9,10_portfolio_direction,4,COBALT is the scale proof point; SKY requires ...,outputs/report_visuals/10_portfolio_direction.png,outputs/report_visuals/10_portfolio_direction.svg,outputs/report_visuals/data/10_portfolio_direc...,"outputs/sales_master.csv, P&L Table.xlsx and M...",Different FY and MAT periods are juxtaposed fo...,clean


,dimension,score,evidence
0,Audience,2,Leadership and FBP audience stated
1,Action,2,Action titles and decision implications
2,Chart fit,2,"Scorecard, bars, heatmaps and additive waterfalls"
3,Accuracy,2,Raw/P&L reconciliation and explicit period cav...
4,Clutter,2,No gridlines; direct labels replace every valu...
5,Attention,2,Grey for context; hue only where the headline ...
6,Accessibility,2,"Signs, position, labels and hatches supplement..."
7,Story,2,Four-page logic from truth to action
8,Medium fit,2,Annotations and sources support distributed sl...
9,Polish,2,"Arial, one left edge for all narrative text, s..."


## Appendix - Verification of `scripts/metrics.py` against the framework

`scripts/metrics.py` is the **certified** analytical base (`certified_for_analysis = True`).
Section 2.2 of `STORYLINE_METRIC_FRAMEWORK.md` keeps it deliberately separate from the
raw/P&L-aligned base that the storyline headlines use, so the two are *expected* to differ.

This appendix does two things:

1. Confirms every quantified framework claim reconciles on the base it is assigned to.
2. Charts where the certified base would change the storyline, so the sensitivity is visible
   rather than buried.

Charts use the `storytelling-with-data` skill (`.claude/skills/storytelling-with-data/`) and are
linted before export. Exports go to `outputs/report_visuals/verification/` so the certified
11-component manifest above is untouched.

> Running these cells calls `swd.use()`, which replaces the global matplotlib rcParams. The
> project palette is snapshotted below and restored in the final cell, so earlier cells can be
> re-run in any order.

In [15]:
# --- Appendix setup: certified-base verification -------------------------
# swd, lint and sys.path are already installed by the theme cell at the top.
import metrics as CERTIFIED

_PROJECT_RCPARAMS = plt.rcParams.copy()   # restored in the final appendix cell
swd.use()

VERIFY_DIR = OUTPUT_DIR / "verification"
VERIFY_DIR.mkdir(parents=True, exist_ok=True)
VERIFY_MANIFEST = []

FY_PRIOR, FY_CURRENT = CERTIFIED.PRIOR_YEAR, CERTIFIED.CURRENT_YEAR
raw_rows = CERTIFIED.sales_df.assign(y=CERTIFIED.sales_df["reporting_month"].dt.year)
cert_rows = raw_rows[raw_rows["certified_for_analysis"]]


def _to(frame, year, col=None, key=None):
    subset = frame if col is None else frame[frame[col] == key]
    return subset.loc[subset["y"] == year, "turnover_keur"].sum()


def to_change(frame, col=None, key=None):
    return _to(frame, FY_CURRENT, col, key) - _to(frame, FY_PRIOR, col, key)


def to_growth(frame, col=None, key=None):
    prior = _to(frame, FY_PRIOR, col, key)
    return (_to(frame, FY_CURRENT, col, key) / prior - 1) * 100 if prior else float("nan")


def save_verification(fig, visual_id, headline, caveat):
    """Lint, then export PNG + SVG + register in the appendix manifest."""
    problems = lint.check(fig)
    errors = [p for p in problems if p.severity == "error"]
    if errors:
        raise AssertionError(f"{visual_id}: " + "; ".join(str(e) for e in errors))
    for suffix in ("png", "svg"):
        fig.savefig(VERIFY_DIR / f"{visual_id}.{suffix}", bbox_inches="tight",
                    facecolor="white", dpi=150)
    VERIFY_MANIFEST.append({
        "visual_id": visual_id,
        "headline": headline,
        "png": f"outputs/report_visuals/verification/{visual_id}.png",
        "svg": f"outputs/report_visuals/verification/{visual_id}.svg",
        "source": "scripts/metrics.py (certified base) vs outputs/sales_master.csv (raw base)",
        "caveat": caveat,
        "lint_warnings": "; ".join(str(p) for p in problems) or "none",
    })
    display(fig)
    plt.close(fig)


# --- the exact checks -----------------------------------------------------
pnl_total = CERTIFIED.pnl_metrics_df.query("brand.str.lower() == 'total'").iloc[0]
scc_prior = pnl_total.prior_supply_chain_cost_eur / pnl_total.prior_turnover_eur
scc_current = pnl_total.current_supply_chain_cost_eur / pnl_total.current_turnover_eur

EXACT_CHECKS = [
    ("2.1 raw GSV FY2023", 371528.3, raw_rows.query("y == @FY_PRIOR").gsv_keur.sum(), 0.05),
    ("2.1 raw GSV FY2024", 349247.6, raw_rows.query("y == @FY_CURRENT").gsv_keur.sum(), 0.05),
    ("2.1 raw TO FY2023", 290066.9, raw_rows.query("y == @FY_PRIOR").turnover_keur.sum(), 0.05),
    ("2.1 raw TO FY2024", 270504.3, raw_rows.query("y == @FY_CURRENT").turnover_keur.sum(), 0.05),
    ("2.1 raw Discount FY2023", 81461.4, raw_rows.query("y == @FY_PRIOR").discount_keur.sum(), 0.05),
    ("2.1 raw Discount FY2024", 78743.3, raw_rows.query("y == @FY_CURRENT").discount_keur.sum(), 0.05),
    ("2.1 TO>GSV rows FY2023 TO", 2574.5,
     _to(raw_rows, FY_PRIOR) - _to(cert_rows, FY_PRIOR), 0.05),
    ("2.1 TO>GSV rows FY2024 TO", 3173.0,
     _to(raw_rows, FY_CURRENT) - _to(cert_rows, FY_CURRENT), 0.05),
    ("4.6 Gross Profit FY2023", 138723.9, pnl_total.prior_gross_profit_eur, 0.1),
    ("4.6 Gross Profit FY2024", 128421.7, pnl_total.current_gross_profit_eur, 0.1),
    ("4.6 Gross Profit change", -10302.1, pnl_total.gross_profit_change_eur, 0.1),
    ("4.6 Gross Margin FY2023 %", 47.82, pnl_total.prior_gross_margin_pct * 100, 0.01),
    ("4.6 Gross Margin FY2024 %", 47.47, pnl_total.current_gross_margin_pct * 100, 0.01),
    ("4.6 Gross Margin movement bps", -35, pnl_total.gross_margin_movement_bps, 0.6),
    ("4.6 Marketing FY2023", 58000.0, pnl_total.prior_marketing_expense_eur, 0.1),
    ("4.6 Marketing FY2024", 42500.0, pnl_total.current_marketing_expense_eur, 0.1),
    ("4.6 Marketing change", -15500.0, pnl_total.marketing_expense_change_eur, 0.1),
    ("4.6 PBO FY2023", 80723.9, pnl_total.prior_pbo_eur, 0.1),
    ("4.6 PBO FY2024", 85921.7, pnl_total.current_pbo_eur, 0.1),
    ("4.6 PBO change", 5197.9, pnl_total.pbo_change_eur, 0.1),
    ("4.6 PBO margin FY2023 %", 27.83, pnl_total.prior_pbo_margin_pct * 100, 0.01),
    ("4.6 PBO margin FY2024 %", 31.76, pnl_total.current_pbo_margin_pct * 100, 0.01),
    ("4.6 PBO margin movement bps", 393, pnl_total.pbo_margin_movement_bps, 1.0),
    ("4.6 PBO bridge residual", 0.0, pnl_total.pbo_bridge_check_eur, 1e-6),
    ("4.6 Supply chain cost change", -9260.0, pnl_total.supply_chain_cost_change_eur, 15),
    ("4.6 Supply Chain Cost/TO FY2023 %", 52.18, scc_prior * 100, 0.01),
    ("4.6 Supply Chain Cost/TO FY2024 %", 52.53, scc_current * 100, 0.01),
    ("4.6 Supply Chain Cost/TO movement bps", 35, (scc_current - scc_prior) * 10_000, 0.6),
]

_market = CERTIFIED.market_metrics_df


def _mkt(channel, brand=None, manufacturer=None):
    rows = _market[_market["channel"].eq(channel)]
    rows = rows[rows["brand"].isna()] if brand is None else rows[rows["brand"].eq(brand)]
    if manufacturer:
        rows = rows[rows["manufacturer"].eq(manufacturer)]
    return rows.iloc[0]


EXACT_CHECKS += [
    ("4.7 Category growth %", -1.3, _mkt("Total", manufacturer="Category").market_value_growth_pct * 100, 0.05),
    ("4.7 UniSweet growth %", -4.3, _mkt("Total", manufacturer="UNISWEET").market_value_growth_pct * 100, 0.05),
    ("4.7 UniSweet share movement pp", -1.97, _mkt("Total", manufacturer="UNISWEET").share_movement_pp, 0.01),
    ("4.7 OLIVE share movement pp", -2.98, _mkt("Total", brand="OLIVE").share_movement_pp, 0.01),
    ("4.7 SKY share movement pp", 0.45, _mkt("Total", brand="SKY").share_movement_pp, 0.01),
    ("4.7 COBALT share movement pp", 0.56, _mkt("Total", brand="COBALT").share_movement_pp, 0.01),
    ("4.7 DT UniSweet share movement pp", -0.28, _mkt("DT", manufacturer="UNISWEET").share_movement_pp, 0.01),
    ("4.7 MT UniSweet share movement pp", -6.07, _mkt("MT", manufacturer="UNISWEET").share_movement_pp, 0.01),
    ("4.7 NAVY DT share movement pp", 0.84, _mkt("DT", brand="NAVY").share_movement_pp, 0.01),
    ("4.7 LILAC MT share movement pp", 5.53, _mkt("MT", brand="LILAC").share_movement_pp, 0.01),
    ("4.7 OLIVE-MT share movement pp", -7.88, _mkt("MT", brand="OLIVE").share_movement_pp, 0.01),
]

verification = pd.DataFrame(
    [{"check": name, "framework": stated, "computed": actual,
      "difference": actual - stated, "status": "PASS" if abs(actual - stated) <= tol else "FAIL"}
     for name, stated, actual, tol in EXACT_CHECKS]
)
CHECKS_PASSED = int((verification["status"] == "PASS").sum())
CHECKS_TOTAL = len(verification)

market_recon_ok = (
    int((_market["sales_value_gain_loss_matches_source"] == False).sum()) == 0
    and int((_market["share_gain_loss_matches_source"] == False).sum()) == 0
)
assert market_recon_ok, "Market source gain/loss columns no longer reconcile"
assert CHECKS_PASSED == CHECKS_TOTAL, verification.query("status == 'FAIL'").to_string()

print(f"{CHECKS_PASSED}/{CHECKS_TOTAL} exact checks pass; "
      f"market source gain/loss reconciles on all rows")
display(verification.style.format({"framework": "{:,.2f}", "computed": "{:,.2f}",
                                   "difference": "{:+,.4f}"}).hide(axis="index"))

39/39 exact checks pass; market source gain/loss reconciles on all rows


check,framework,computed,difference,status
2.1 raw GSV FY2023,"371,528.30","371,528.30",+0.0000,PASS
2.1 raw GSV FY2024,"349,247.60","349,247.60",+0.0000,PASS
2.1 raw TO FY2023,"290,066.90","290,066.90",+0.0000,PASS
2.1 raw TO FY2024,"270,504.30","270,504.30",+0.0000,PASS
2.1 raw Discount FY2023,"81,461.40","81,461.40",+0.0000,PASS
2.1 raw Discount FY2024,"78,743.30","78,743.30",-0.0000,PASS
2.1 TO>GSV rows FY2023 TO,"2,574.50","2,574.50",+0.0000,PASS
2.1 TO>GSV rows FY2024 TO,"3,173.00","3,173.00",+0.0000,PASS
4.6 Gross Profit FY2023,"138,723.90","138,723.85",-0.0450,PASS
4.6 Gross Profit FY2024,"128,421.70","128,421.75",+0.0450,PASS


In [16]:
# V1 - verification result as a single number
fig, ax = swd.figure(
    "Every framework figure reconciles on the base it is assigned to",
    subtitle="Exact checks of STORYLINE_METRIC_FRAMEWORK.md against scripts/metrics.py",
    source=("Checks: raw-base reconciliation, P&L levels, margins, PBO bridge, supply-chain rate, "
            "Market growth and share  |  Market source gain/loss ties on every row"),
    figsize=(13.3, 5.6))
swd.bignum(ax, f"{CHECKS_PASSED} / {CHECKS_TOTAL}",
           "exact checks pass - the certified engine's formulas match section 4 of the framework")
save_verification(
    fig, "V1_verification_result",
    "Every framework figure reconciles on the base it is assigned to",
    "Confirms formula compliance and reconciliation, not that the raw base is validated; "
    "TO > GSV rows remain open")

<Figure size 1995x840 with 1 Axes>

In [17]:
# V2 - would the certified base change the growth rates the storyline quotes?
# OLIVE (-7.3%) and SKY (-7.4%) sit 0.1pp apart and within 0.7pp of Total; on a
# slopegraph their labels overlap illegibly, so they are quoted in the footnote.
cuts = [("Total", None, None), ("COBALT", "brand_name", "COBALT"),
        ("DT", "channel_code", "DT"), ("MT", "channel_code", "MT")]
growth_pairs = {label: (to_growth(raw_rows, col, key), to_growth(cert_rows, col, key))
                for label, col, key in cuts}

fig, ax = swd.figure(
    "The certified base makes every decline look steeper, most of all in MT",
    subtitle=f"FY{FY_CURRENT} turnover growth %, same formula on two bases",
    source=("Source: scripts/metrics.py  |  Certified base excludes rows flagged TO > GSV "
            "(2,574.5 kEUR of FY2023 TO, 3,173.0 kEUR of FY2024 TO). "
            "OLIVE -7.3% -> -7.5% and SKY -7.4% -> -7.6% omitted: they overlap Total"))
swd.slopegraph(ax, "Raw / P&L-aligned\n(storyline base)", "Certified\n(metrics.py base)",
               growth_pairs, highlight={"MT", "COBALT"}, unit="%", decimals=1)
save_verification(
    fig, "V2_growth_base_sensitivity",
    "The certified base makes every decline look steeper, most of all in MT",
    "Two different bases, not two periods; the framework quotes the raw base throughout")

<Figure size 1920x1080 with 1 Axes>

In [18]:
# V3 - which drivers actually move when the base changes
entities = [("Macarons", "customer_name"), ("MT", "channel_code"), ("OLIVE", "brand_name"),
            ("PACK 1.1KG", "product_name"), ("COBALT", "brand_name"),
            ("POUCH 900GR", "product_name"), ("POUCH 400GR", "product_name"),
            ("Candies", "customer_name"), ("Bliss", "customer_name"),
            ("DT", "channel_code"), ("POUCH 100GR", "product_name")]
base_gap = [(name, (to_change(cert_rows, col, name) - to_change(raw_rows, col, name)) / 1000)
            for name, col in entities]

fig, ax = swd.figure(
    "Macarons is the one driver whose size changes materially with the base",
    subtitle=("Certified minus raw FY2024 turnover change, EURm "
              "(negative = the certified base shows a bigger loss)"),
    source=("Source: scripts/metrics.py  |  Sensitivity, not a correction. On the certified base "
            "Macarons reads -5.60m / -35.7% instead of -4.35m / -25.7%"))
swd.hbar(ax, [row[0] for row in base_gap], [row[1] for row in base_gap],
         highlight={"Macarons", "MT"}, decimals=2, sort=True)
save_verification(
    fig, "V3_driver_base_gap",
    "Macarons is the one driver whose size changes materially with the base",
    "Framework guardrail 5 still holds on both bases: this is concentration, not attrition")

<Figure size 1920x1080 with 1 Axes>

In [19]:
# V4 - why H2 concentration is the most base-sensitive claim
flagged = raw_rows[~raw_rows["certified_for_analysis"]]
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
flagged_by_month = []
for month in range(1, 13):
    subset = flagged[flagged["reporting_month"].dt.month == month]
    flagged_by_month.append(
        (subset.loc[subset["y"] == FY_CURRENT, "turnover_keur"].sum()
         - subset.loc[subset["y"] == FY_PRIOR, "turnover_keur"].sum()) / 1000)

h1_effect, h2_effect = sum(flagged_by_month[:6]), sum(flagged_by_month[6:])

fig, ax = swd.figure(
    f"Flagged rows add {h1_effect:.1f}m to H1 and take {abs(h2_effect):.1f}m off H2, "
    f"softening the H2 story",
    subtitle="FY2024 vs FY2023 turnover change contributed by rows excluded from the certified base, EURm",
    source=("Source: scripts/metrics.py  |  Positive = the flagged rows make that month look better "
            "on the raw base. H2 share of the FY decline is 95.7% raw, 90.2% certified"))
swd.bar(ax, month_names, flagged_by_month, highlight={"Mar", "Jul", "Dec"}, decimals=2)
save_verification(
    fig, "V4_flagged_rows_by_month",
    "Flagged rows flatter H1 and depress H2, softening the H2 concentration claim",
    "Framework guardrail 6 still applies: monthly sell-in does not establish seasonal causality")

<Figure size 1920x1080 with 1 Axes>

In [20]:
# --- Appendix export manifest and rcParams restore ------------------------
verify_manifest = pd.DataFrame(VERIFY_MANIFEST)
verify_manifest.to_csv(VERIFY_DIR / "verification_manifest.csv", index=False)
verification.to_csv(VERIFY_DIR / "framework_check_results.csv", index=False)

for row in VERIFY_MANIFEST:
    png = PROJECT_ROOT / row["png"]
    with Image.open(png) as image:
        assert image.width > 0 and image.height > 0, png
    svg = PROJECT_ROOT / row["svg"]
    tree = ElementTree.parse(svg)
    assert "Arial" in svg.read_text(encoding="utf-8"), f"{svg} lost the Arial font stack"

plt.rcParams.update(_PROJECT_RCPARAMS)   # hand the project palette back

print(f"{len(VERIFY_MANIFEST)} verification visuals exported to {VERIFY_DIR}")
print(f"lint: {sum(1 for r in VERIFY_MANIFEST if r['lint_warnings'] == 'none')}"
      f"/{len(VERIFY_MANIFEST)} clean")
display(verify_manifest[["visual_id", "headline", "lint_warnings"]])

4 verification visuals exported to /Users/tranvomanhtuan/Documents/05_Business_Case_Studies/SEO-V FBP Case Data/outputs/report_visuals/verification
lint: 4/4 clean


,visual_id,headline,lint_warnings
0,V1_verification_result,Every framework figure reconciles on the base ...,none
1,V2_growth_base_sensitivity,The certified base makes every decline look st...,none
2,V3_driver_base_gap,Macarons is the one driver whose size changes ...,none
3,V4_flagged_rows_by_month,"Flagged rows flatter H1 and depress H2, soften...",none
